# Teste isolado — ARISB-MG (RSS nativo)

Fonte candidata: **ARISB-MG** — Agência Reguladora Intermunicipal de
Saneamento Básico de Minas Gerais, setor Saneamento. Notebook
**descartável** (Fase 1) — sem dispatcher, sem gravar nada.

## Confirmado antes de assumir

O site (`arisb.com.br/portal/noticias`) tem um RSS nativo em
`https://www.arisb.com.br/portal/rss` -- já confirmado manualmente que
responde com XML RSS 2.0 válido (título, descrição, pubDate, link, imagem
embutida na descrição). Esse teste confirma que o **parsing programático**
(via `feedparser`, mesma lib já usada nos outros dispatchers de RSS do
projeto) funciona igual, e que a tag `<img>` dentro da descrição é limpa
corretamente.


In [0]:
%pip install --quiet feedparser beautifulsoup4 lxml
dbutils.library.restartPython()

In [0]:
import feedparser
from bs4 import BeautifulSoup


In [0]:
FEED_URL = "https://www.arisb.com.br/feedrss.xml"


## Etapa 1 — Buscar e parsear o feed


In [0]:
feed = feedparser.parse(FEED_URL)

print(f"Status HTTP: {feed.get('status', '?')}")
print(f"Título do canal: {feed.feed.get('title', '?')!r}")
print(f"Total de itens no feed: {len(feed.entries)}")

sem_titulo = [e for e in feed.entries if not e.get('title')]
sem_data = [e for e in feed.entries if not e.get('published')]
sem_link = [e for e in feed.entries if not e.get('link')]
print(f"Sem título: {len(sem_titulo)}")
print(f"Sem data: {len(sem_data)}")
print(f"Sem link: {len(sem_link)}")


## Etapa 2 — Limpar a descrição (checar se a tag <img> some) e conferir uma amostra


In [0]:
def limpar_descricao(html_bruto: str) -> str:
    soup = BeautifulSoup(html_bruto, "lxml")
    for img in soup.find_all("img"):
        img.decompose()
    return soup.get_text("\n", strip=True)


AMOSTRA = 5
resultados = []

for entry in feed.entries[:AMOSTRA]:
    texto_limpo = limpar_descricao(entry.get("description", ""))
    tem_img_sobrando = "<img" in texto_limpo.lower()
    resultados.append({
        "titulo": entry.get("title"),
        "data": entry.get("published"),
        "link": entry.get("link"),
        "texto": texto_limpo,
        "tem_img_sobrando": tem_img_sobrando,
    })
    print(f"\n[item] {entry.get('title', '?')[:80]}")
    print(f"    -> {len(texto_limpo)} chars de texto limpo. img sobrando? {tem_img_sobrando}")


## Etapa 3 — Mostrar o texto completo do primeiro item, pra inspeção visual


In [0]:
r = resultados[0]

print("=" * 100)
print(f"TÍTULO : {r['titulo']}")
print(f"DATA   : {r['data']}")
print(f"LINK   : {r['link']}")
print(f"TAMANHO: {len(r['texto'])} chars")
print("=" * 100)
print(r["texto"])


## Conclusão da Fase 1

Confirmar: (a) `sem_titulo`/`sem_data`/`sem_link` vazios; (b)
`tem_img_sobrando = False` em todos os itens da amostra (senão, a limpeza
de HTML precisa de ajuste antes de virar produção); (c) texto legível, sem
lixo de tag HTML sobrando.

Se tudo confirmado: **não precisa de notebook de produção novo** — só
adicionar uma entrada nova no `CONFIGS_FONTES` do
`ingest-news-rss-infra.ipynb` (dispatcher de RSS genérico já existente),
com `"feeds": ["https://www.arisb.com.br/portal/rss"]`.
